# 5. Проверка устойчивости спецификаций

**Цель:** вместо несуществующего в публичном API многоагентного интерфейса сопоставить несколько допустимых спецификаций `GraphEVTPipeline`. Таблица явно показывает, какие выводы устойчивы к порогу хвоста, вероятности alarm и persistence.


In [ ]:
from pathlib import Path
from importlib.metadata import version
import numpy as np
import pandas as pd
from graph_evt_agent import EVTConfig, GraphConfig, GraphEVTPipeline, InputConfig

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/fractal_extreme_series.csv").exists())
DATA = ROOT / "data/fractal_extreme_series.csv"
SEED = 42
df = pd.read_csv(DATA)
x = df["value"].to_numpy()
assert np.isfinite(x).all() and x.ndim == 1


In [ ]:
BASELINE_SIZE = int(0.75 * len(x))
specifications = [
    {"role": "sensitive", "tail_quantile": 0.90, "alarm_probability": 0.995, "persistence": 1},
    {"role": "primary", "tail_quantile": 0.95, "alarm_probability": 0.999, "persistence": 3},
    {"role": "conservative", "tail_quantile": 0.975, "alarm_probability": 0.9995, "persistence": 5},
]
rows = []
for spec in specifications:
    evt = EVTConfig(
        baseline_size=BASELINE_SIZE,
        tail_quantile=spec["tail_quantile"],
        alarm_probability=spec["alarm_probability"],
        persistence=spec["persistence"],
    )
    run = GraphEVTPipeline(
        evt,
        graph=GraphConfig(random_state=SEED),
        input_config=InputConfig(missing="error"),
    ).run_detailed(x)
    rows.append({
        **spec,
        "alarm_threshold": run.detection.alarm_threshold,
        "post_baseline_exceedances": int(np.sum(run.detection.indicator[BASELINE_SIZE:] > run.detection.alarm_threshold)),
        "detected": run.detection.detected,
        "first_persistent_index": run.detection.time_index,
        "graph_available": run.graph is not None,
        "ranking_available": run.ranking is not None,
    })
sensitivity = pd.DataFrame(rows)
sensitivity


In [ ]:
artifact_dir = ROOT / "results"
artifact_dir.mkdir(exist_ok=True)
sensitivity.to_csv(artifact_dir / "05_specification_review.csv", index=False)

agreement = {
    "all_detected": bool(sensitivity["detected"].all()),
    "none_detected": bool((~sensitivity["detected"]).all()),
    "same_first_index": bool(sensitivity["first_persistent_index"].nunique(dropna=False) == 1),
    "library_version": version("graph-evt-agent"),
}
agreement


Разные строки — не независимые агенты, а прозрачные статистические спецификации. Расхождение между ними является анализом чувствительности, а совпадение не доказывает истинность. Для одномерного ряда библиотека закономерно не строит граф и ranking.

**Самопроверка:** какая спецификация более чувствительна? Почему перебор настроек после просмотра результата может завысить уверенность? Какие дополнительные baseline-сценарии следует проверить?
